**Group O Members**
**RAG-BASED SAFE GUARDING COMPANION**

| Student Name | Registration Number | Student ID Number |
|---|---|---|
| Namulinde Jean Madrine L | 24/U/09341/EVE | 2400709341 |
| Wasswa Job Kitandwe | 24/U/11882/EVE | 2400711882 |
| Tugume Jonathan | 24/U/11526/EVE | 2400711526 |
| Otieno Fidel Jone | 24/U/25270/EVE | 2400725270 |
| Atugonza Mathias | 24/U/0273 | 2400700273 |

## Project Overview

The Safeguarding Companion is a RAG-based chatbot built to help Makerere University students  especially persons with disabilities (PWDs)  understand and act on university safeguarding policies. Despite strong policies on sexual harassment, disability rights, and safeguarding existing at the university, most students can't access or understand them because they are dense legal documents, so the chatbot bridges that gap by retrieving relevant chunks from official policy documents and generating grounded, citation backed answers in plain simple language meaning every answer traces back to a real document and nothing is made up. Accessibility is central to the design, with screen reader support.

# 🛡️ Safeguarding Companion
### RAG-Based Policy Q&A System — Makerere University

**Pipeline:**
1. Install libraries
2. Paste Groq API key
3. Download pre-built chunks from GitHub
4. Load embedding model
5. Retrieval functions
6. Generation with Groq (Llama 3)
7. Test + interactive demo

---

## Step 1 — Install Libraries

In [ ]:
!pip install -q sentence-transformers scikit-learn nltk requests numpy pandas

##  Groq API Key

for answer generation

In [ ]:
# ──  GROQ API KEY ─────────────────────────────────────────
GROQ_API_KEY = ""
# ─────────────────────────────────────────────────────────────────────────

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
GROQ_MODEL   = "llama-3.1-8b-instant"

print("✅ API key set." if GROQ_API_KEY != "paste-your-key-here" else "⚠️  Please paste your Groq API key above!")

✅ API key set.


## Step 3 — Imports and Configuration

In [ ]:
import os, re, requests
import numpy as np
import pandas as pd
import nltk
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

TOP_K                = 7
SIMILARITY_THRESHOLD = 0.20

STOP_WORDS = {"the","and","for","are","that","this","with","how","what","who","can","you","was","has","have","been","hello","hi","hey","please","thanks","thank"}
GREETINGS  = {"hi","hello","hey","hie","howdy","good morning","good afternoon","good evening","greetings"}

GREETING_RESPONSE = "Hello! Welcome to the Safeguarding Companion.\n\nI can help you understand university policies on safeguarding, disability rights, sexual harassment, and more.\n\nTry asking:\n- How do I report harassment?\n- What rights do students with disabilities have?\n- What is the HIV/AIDS policy?"

SKIP_PHRASES = ["there is no documented","current evidence","principles underpinning","it should be noted","as quasi-judicial","no current evidence"]

print("✅ Imports and config ready.")

✅ Imports and config ready.


## Step 4 — Download Pre-built Chunks from GitHub

Downloads `policy_chunks.csv` and `chunk_embeddings.npy` directly from the project repo.  
No PDF processing or OCR needed — saves several minutes.

In [ ]:
GITHUB_RAW_BASE    = "https://raw.githubusercontent.com/madrinejean123/MACHINE-LEARNING--CHATBOT-GROUP-O/madrine"
CHUNK_CSV_URL      = f"{GITHUB_RAW_BASE}/policy_chunks.csv"
EMBEDDINGS_NPY_URL = f"{GITHUB_RAW_BASE}/chunk_embeddings.npy"

def download_file(url, local_path):
    print(f"Downloading {os.path.basename(local_path)} ...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(local_path, "wb") as f:
        f.write(r.content)
    print(f"  ✅ Saved — {len(r.content)//1024} KB")

download_file(CHUNK_CSV_URL,      "policy_chunks.csv")
download_file(EMBEDDINGS_NPY_URL, "chunk_embeddings.npy")

df         = pd.read_csv("policy_chunks.csv")
embeddings = np.load("chunk_embeddings.npy")

print(f"\n✅ Loaded {len(df)} policy chunks, embeddings shape: {embeddings.shape}")
print("\nSource documents:")
for src in df['source_document'].unique():
    print(f"  📄 {src} — {len(df[df['source_document']==src])} chunks")

  ✅ Saved — 516 KB
  ✅ Saved — 699 KB

✅ Loaded 466 policy chunks, embeddings shape: (466, 384)

Source documents:
  📄 FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf — 210 chunks
  📄 HIV_AIDS_Policy.pdf — 33 chunks
  📄 Makerere-Policy-on-Persons-Living-With-Disabilities.pdf — 66 chunks
  📄 Makerere-Safeguarding-Policy.pdf — 61 chunks
  📄 Policy-and-Regulations-Against-Sexual-Harassment-2018.pdf — 38 chunks
  📄 UTAMU-Disability-Policy.pdf — 58 chunks


## Step 5 — Load Embedding Model

`all-MiniLM-L6-v2` converts questions and policy text into vectors for semantic similarity search.

In [ ]:
print("Loading embedding model...")
emb_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model ready.")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model ready.


## Step 6 — Retrieval Functions

Three stages: query expansion → keyword filter → cosine similarity search

In [ ]:
QUERY_EXPANSIONS = {
    "harass":    "report complaint procedure steps lodge file officer committee",
    "assault":   "report complaint procedure steps lodge file officer committee",
    "disabilit": "rights accommodation access support services equal opportunity",
    "disabled":  "rights accommodation access support services equal opportunity",
    "hiv":       "rights confidentiality treatment support non-discrimination policy",
    "aids":      "rights confidentiality treatment support non-discrimination policy",
    "report":    "complaint procedure steps lodge file officer committee",
    "complain":  "complaint procedure steps lodge file officer committee",
    "right":     "rights responsibilities protection policy entitlement",
    "protect":   "safeguarding protection rights policy procedure",
}

ACTION_WORDS = ["report","complain","lodge","file","contact","procedure","steps","committee","officer","directorate","submit","notify","support","rights","entitled","must","shall","access"]

def expand_query(query):
    q_lower = query.lower()
    extras  = set()
    for trigger, expansion in QUERY_EXPANSIONS.items():
        if trigger in q_lower:
            extras.update(expansion.split())
    return query + " " + " ".join(extras) if extras else query

def keyword_filter(df, query):
    keywords = [w.lower() for w in re.findall(r"\b\w+\b", query) if len(w) > 2 and w.lower() not in STOP_WORDS]
    if not keywords: return df
    mask     = df["text"].apply(lambda x: any(k in str(x).lower() for k in keywords))
    filtered = df[mask]
    return filtered if len(filtered) > 0 else df

def retrieve_top_k(query, k=TOP_K, threshold=SIMILARITY_THRESHOLD):
    expanded = expand_query(query)
    filtered = keyword_filter(df, query)
    indices  = filtered.index.tolist()
    f_embeds = embeddings[indices]
    q_vec    = emb_model.encode([expanded], normalize_embeddings=True)
    scores   = cosine_similarity(q_vec, f_embeds)[0]
    sorted_i = np.argsort(scores)[::-1]
    top_i    = [i for i in sorted_i[:k] if scores[i] >= threshold]
    if not top_i: top_i = sorted_i[:5]
    results = filtered.iloc[top_i].copy()
    results["similarity_score"] = scores[top_i]
    results["action_boost"]     = results["text"].apply(lambda t: sum(1 for w in ACTION_WORDS if w in str(t).lower()))
    results = results.sort_values(by=["action_boost","similarity_score"], ascending=[False,False]).drop(columns=["action_boost"])
    return results

print("✅ Retrieval functions ready.")

✅ Retrieval functions ready.


## Step 7 — Answer Generation with Groq (Llama 3)

Sends retrieved chunks to Groq with a strict prompt: only rewrite what is given, fix OCR errors, use plain English.

In [ ]:
def nice_source_name(raw):
    return raw.replace(".pdf","").replace("-"," ").replace("_"," ").strip()

def is_greeting(text):
    cleaned = text.strip().lower().rstrip("!.,?")
    words   = cleaned.split()
    return cleaned in GREETINGS or (len(words) <= 3 and words[0] in GREETINGS)

def polish_with_groq(question, raw_policy_text):
    if not GROQ_API_KEY or GROQ_API_KEY == "paste-your-key-here":
        return None
    system_prompt = (
        "You are a helpful university safeguarding assistant for Makerere University students. "
        "Explain university policies in plain, simple English.\n"
        "Rules:\n"
        "1. Fix OCR merged words (e.g. 'mustbe' -> 'must be').\n"
        "2. Use numbered list (1. 2. 3.) for procedures.\n"
        "3. Use bullet points for general info.\n"
        "4. Use ONLY the provided policy text — do not add new information.\n"
        "5. Remove repeated sentences. Keep it short and friendly.\n"
        "6. End with: 'For more help, contact the Directorate of Gender Mainstreaming at gendermainstreaming@mak.ac.ug or call +256 (0)414 532 631.'"
    )
    user_prompt = f"A student asked: \"{question}\"\n\nRaw policy text:\n---\n{raw_policy_text[:3000]}\n---\n\nWrite a clear, simple answer:"
    try:
        r = requests.post(
            GROQ_API_URL,
            headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
            json={"model": GROQ_MODEL, "messages": [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}], "max_tokens": 600, "temperature": 0.3},
            timeout=30,
        )
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip() or None
    except Exception as e:
        print(f"Groq error: {e}"); return None

def fallback_format(retrieved, query=""):
    grouped = {}
    for _, row in retrieved.iterrows():
        src = nice_source_name(row["source_document"])
        grouped.setdefault(src,[]).append(row["text"].strip())
    parts, total = [], 0
    for src, texts in grouped.items():
        parts.append(f"\n**{src}**")
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', " ".join(texts)) if len(s.split())>=6]
        sentences = [s for s in sentences if not any(p in s.lower() for p in SKIP_PHRASES)]
        for s in sentences[:6]:
            if not s.endswith(('.','!','?')): s += '.'
            parts.append(f"- {s}"); total += 1
    if total == 0:
        return "Could not extract clear steps.\nContact: 📞 +256 (0)414 532 631 · 📧 gendermainstreaming@mak.ac.ug"
    return (f"Policies say about **{query}**:\n" if query else "") + "\n".join(parts)

def generate_answer(question):
    if is_greeting(question):
        return GREETING_RESPONSE
    retrieved = retrieve_top_k(question)
    if retrieved.empty:
        return "Could not find information about that.\nContact: 📞 +256 (0)414 532 631"
    raw_policy_text = "\n\n".join(f"[Source: {nice_source_name(row['source_document'])}]\n{row['text'].strip()}" for _, row in retrieved.iterrows())
    groq_result = polish_with_groq(question, raw_policy_text)
    answer      = groq_result if groq_result else fallback_format(retrieved, query=question)
    sources     = " | ".join(nice_source_name(s) for s in retrieved["source_document"].unique())
    return answer + f"\n\n📄 Sources: {sources}"

print("✅ Generation functions ready.")

✅ Generation functions ready.


## Step 8 — Test with Example Questions

In [ ]:
test_questions = [
    "How do I report sexual harassment?",
    "What rights do students with disabilities have?",
    "What is the HIV/AIDS policy?",
]

for question in test_questions:
    print("=" * 60)
    print(f"❓ {question}")
    print("-" * 60)
    print(generate_answer(question))
    print()

❓ How do I report sexual harassment?
------------------------------------------------------------
If you've experienced sexual harassment, here's what you should do:

**Step 1: Seek help**
- Go to a hospital or a medical facility for help if you've been physically harmed.
- Keep a record of any medical treatment you receive.
- Learn about our university's policy on sexual harassment.

**Step 2: Report the incident**
- If you're unsure about what to do, contact the Directorate of Gender Mainstreaming for assistance.
- If you feel safe, tell the person who harassed you to stop. Be direct and clear.
- Keep evidence of the incident, such as messages, emails, or witness statements.

**Step 3: File a complaint**
- You can report the incident through a complaint box or by calling our telephone hotline.
- If you want to remain anonymous, your complaint will be investigated first to confirm its authenticity.
- However, if you want to pursue a remedy, you'll need to be identified to the person w

## Step 9 — Interactive Demo

Change `your_question` to anything you want to ask.

In [ ]:
# ── TYPE YOUR QUESTION HERE ───────────────────────────────────────────────
your_question = "How do I file a complaint?"
# ─────────────────────────────────────────────────────────────────────────

print(f"❓ {your_question}")
print("-" * 60)
print(generate_answer(your_question))

❓ How do I file a complaint?
------------------------------------------------------------
If you've experienced sexual harassment, here's what you can do:

1. Seek medical help if you've been physically harmed. Keep a record of your treatment.
2. Familiarize yourself with our University Policy and Regulations Against Sexual Harassment.
3. If you're unsure about what to do, contact the Directorate of Gender Mainstreaming for assistance.

To file a complaint:

1. You can report your complaint in person, by phone, or through our online reporting system.
2. If you prefer to remain anonymous, you can report your complaint through a complaint box or phone hotline. However, please note that anonymous complaints will be investigated first to determine their authenticity.
3. If you choose to remain anonymous, you may not be able to pursue a remedy through our procedures.

To make a formal complaint, you'll need to:

1. Write and sign a complaint with the Gender Mainstreaming Directorate or any 

## Step 10 — Retrieval Inspection

See exactly which chunks were retrieved and their similarity scores.

In [ ]:
inspect_question = "What support is available for students with disabilities?"
results = retrieve_top_k(inspect_question)

print(f"Top {len(results)} chunks for: '{inspect_question}'\n")
for i, (_, row) in enumerate(results.iterrows(), 1):
    print(f"Chunk {i} — {nice_source_name(row['source_document'])}")
    print(f"  Similarity: {row['similarity_score']:.3f}")
    print(f"  Text: {row['text'][:200]}...\n")

Top 7 chunks for: 'What support is available for students with disabilities?'

Chunk 1 — Makerere Policy on Persons Living With Disabilities
  Similarity: 0.679
  Text: The University therefore is committed to respect, promote and protect the rights of persons living with disabilities to work and study on an equitable basis with other members of the University commun...

Chunk 2 — UTAMU Disability Policy
  Similarity: 0.743
  Text: To develop linkages with local and international communities, gove rnment and NGOs to support and facilitate full inclusion of students with disabilities in all University activities. RIGHTS AND RESPO...

Chunk 3 — Makerere Policy on Persons Living With Disabilities
  Similarity: 0.697
  Text: Information on services to assist students with disabilities will be publicised on the University web site. Support for Students with a Disability As part of academic support services, the University ...

Chunk 4 — Makerere Policy on Persons Living With Disabilities
  

---
## Summary

| Component | Technology |
|---|---|
| Embeddings | `all-MiniLM-L6-v2` (SentenceTransformers) |
| Semantic search | Cosine similarity (scikit-learn) |
| Answer generation | Groq — Llama 3.1 8B Instant |
| Policy sources | 6 official Makerere University documents |
| Deployment | Hugging Face Spaces (Streamlit) |

**Live app:** https://madrine-safeguarding-companion.hf.space